# Anarkali: real-world test

Scores a finished run on pages outside its own dataset. Nothing is trained.

1. **Random web**: one fresh WARC from Common Crawl, every kind of page, only from sites that
   are not in the run's dataset. page_type is scored where the page's own schema.org gives a label;
   on all pages we report how often the model is confident and what it predicts.
2. **Human labels**: the Zyte product extraction benchmark (MIT, labelled by people, not by
   schema.org) for price_field and in_stock.
3. **Review sample**: random pages with the model's answer, saved for a person to check.

Results go to `RUN_NAME/realworld/` on HF. Kaggle: GPU on (CPU also works, slower), Internet on,
secret HF_TOKEN attached.


In [ ]:
RUN_NAME = "run2"
HF_REPO = "toufiqqureshi651/anarkali-web"
REPO_URL = "https://github.com/ToufiqQureshi/anarkali.git"
BRANCH = "main"
CRAWL = "latest"          # a crawl id, or "latest"
SEED = 7                  # different from the dataset build, so different WARC files
MAX_PAGES = 6000          # pages scored from the random web
MAX_PER_HOST = 3
REVIEW_PAGES = 150

In [ ]:
import os, subprocess, sys, json, random, time
from collections import Counter, defaultdict
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO = WORK / "anarkali"
OUT = WORK / "realworld"
OUT.mkdir(exist_ok=True)
LOG = OUT / "log.txt"

def sh(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def say(*parts):
    text = " ".join(str(p) for p in parts)
    print(text, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(text + "\n")

if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
sh('pip install -q -e ".[train]" warcio requests', cwd=REPO)
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "scripts"))

def _hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        raise RuntimeError("add the HF_TOKEN secret (Kaggle: Add-ons -> Secrets; Colab: key icon on the left) and run again")
    return token

import torch
from huggingface_hub import HfApi, snapshot_download
HF_TOKEN = _hf_token()
hf = HfApi(token=HF_TOKEN)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
say("Hugging Face user:", hf.whoami()["name"], "| device:", DEVICE)

In [ ]:
from anarkali.engine import Engine
from anarkali.typed import question_candidates
from anarkali.web import (IN_STOCK_QUESTION, PAGE_TYPE_QUESTION, PRICE_QUESTION_TEXT, MAX_PRICE_OPTIONS,
                          page_decisions, page_state, parse_page, parse_amount, price_candidates)

got = Path(snapshot_download(HF_REPO, token=HF_TOKEN, allow_patterns=[
    f"{RUN_NAME}/web-run/best.pt", f"{RUN_NAME}/web-run/test-report.json", f"{RUN_NAME}/web-decisions-v0/*.jsonl"]))
report = json.loads((got / RUN_NAME / "web-run" / "test-report.json").read_text())
engine = Engine.load(str(got / RUN_NAME / "web-run" / "best.pt"), device=DEVICE)
engine.temperature_by_type = report["temperature_by_type"]   # the same calibration the release uses

# every host in the run's dataset: the random-web test only uses sites the model has never seen
known_hosts = set()
for split in ("train", "development", "calibration", "test"):
    for line in open(got / RUN_NAME / "web-decisions-v0" / f"{split}.jsonl", encoding="utf-8"):
        known_hosts.add(json.loads(line)["source_group"])
say("hosts in the run's dataset:", len(known_hosts), "| temperatures:", engine.temperature_by_type)

PT_KIND, PT_TEXT, PT_OPTIONS = question_candidates(PAGE_TYPE_QUESTION)
ST_KIND, ST_TEXT, ST_OPTIONS = question_candidates(IN_STOCK_QUESTION)

def score(items):
    """Probabilities for (kind, state, text, candidates) items, in batches."""
    out = []
    for start in range(0, len(items), 256):
        probs, _ = engine.score(items[start:start + 256])
        out.extend(probs)
    return out

def summary(pairs):
    """pairs: (correct: bool, confidence: float). Accuracy, and coverage/accuracy at confidence >= 0.9."""
    n = len(pairs)
    sure = [ok for ok, conf in pairs if conf >= 0.9]
    return {"n": n, "accuracy": round(sum(ok for ok, _ in pairs) / n, 4) if n else None,
            "confident_0.9_coverage": round(len(sure) / n, 4) if n else None,
            "confident_0.9_accuracy": round(sum(sure) / len(sure), 4) if sure else None}

## 1. Random web (sites the model never saw)


In [ ]:
import requests
from build_web_decisions import host_of, html_records, latest_crawl, warc_urls

session = requests.Session()
session.headers["User-Agent"] = "anarkali-realworld-test/0 (research evaluation)"
crawl = latest_crawl(session) if CRAWL == "latest" else CRAWL
say("crawl:", crawl)

pages, per_host, skipped = [], Counter(), Counter()
for url in warc_urls(session, crawl, 3, SEED):
    if len(pages) >= MAX_PAGES:
        break
    say("streaming", url.rsplit("/", 1)[-1])
    try:
        response = session.get(url, stream=True, timeout=300)
        response.raise_for_status()
        for page_url, html in html_records(response.raw, 40_000):
            host = host_of(page_url)
            if not host or host in known_hosts:
                skipped["known_or_empty_host"] += 1
                continue
            if per_host[host] >= MAX_PER_HOST:
                continue
            state = page_state(html, page_url)
            if len(state["text"]) < 200:
                skipped["too_little_text"] += 1
                continue
            rows = {r["case_id"].rsplit("::", 1)[-1]: r for r in page_decisions(html, page_url, host)}
            pages.append({"url": page_url, "host": host, "state": state, "rows": rows})
            per_host[host] += 1
            if len(pages) >= MAX_PAGES:
                break
    except Exception as exc:   # one broken WARC must not stop the test
        say("WARC failed:", repr(exc)[:200])
say("pages:", len(pages), "from", len(per_host), "new sites | skipped:", dict(skipped))

started = time.time()
pt_probs = score([(PT_KIND, p["state"], PT_TEXT, PT_OPTIONS) for p in pages])
say(f"page_type scored in {time.time() - started:.0f}s")
ids = [o["id"] for o in PT_OPTIONS]
for p, probs in zip(pages, pt_probs):
    best = max(range(len(ids)), key=probs.__getitem__)
    p["page_type"], p["page_type_conf"] = ids[best], probs[best]

web = {"crawl": crawl, "pages": len(pages), "sites": len(per_host)}
web["predicted_page_type"] = dict(Counter(p["page_type"] for p in pages))
web["confident_0.9_share_all_pages"] = round(sum(p["page_type_conf"] >= 0.9 for p in pages) / len(pages), 4)
labelled = [p for p in pages if "page_type" in p["rows"]]
truth = lambda row: [c["id"] for c in row["candidates"]][max(range(len(row["target"])), key=row["target"].__getitem__)]
web["page_type_vs_schema"] = summary([(p["page_type"] == truth(p["rows"]["page_type"]), p["page_type_conf"])
                                      for p in labelled])
web["page_type_vs_schema"]["majority_baseline"] = (
    round(Counter(truth(p["rows"]["page_type"]) for p in labelled).most_common(1)[0][1] / len(labelled), 4)
    if labelled else None)
per_label = defaultdict(list)
for p in labelled:
    per_label[truth(p["rows"]["page_type"])].append((p["page_type"] == truth(p["rows"]["page_type"]), p["page_type_conf"]))
web["page_type_by_true_label"] = {k: summary(v) for k, v in sorted(per_label.items())}

for key, kind in (("in_stock", ST_KIND), ("price_field", "choice")):
    have = [p for p in pages if key in p["rows"]]
    if not have:
        continue
    probs = score([(kind, p["rows"][key]["state"], p["rows"][key]["question"], p["rows"][key]["candidates"])
                   for p in have])
    pairs = []
    for p, pr in zip(have, probs):
        best = max(range(len(pr)), key=pr.__getitem__)
        pairs.append((p["rows"][key]["target"][best] > 0, pr[best]))
    web[f"{key}_vs_schema"] = summary(pairs)
say(json.dumps(web, indent=1))

## 2. Human labels: Zyte product extraction benchmark


In [ ]:
import gzip
ZYTE = WORK / "product-extraction-benchmark"
if not ZYTE.exists():
    sh(f"git clone --depth 1 https://github.com/scrapinghub/product-extraction-benchmark {ZYTE}")
truth_by_id = json.loads((ZYTE / "dataset" / "ground-truth.json").read_text(encoding="utf-8"))

stock_items, stock_truth, price_items, price_gold, missing = [], [], [], [], Counter()
for page_id, gt in truth_by_id.items():
    path = ZYTE / "dataset" / "html" / f"{page_id}.html.gz"
    if not path.exists():
        missing["html"] += 1
        continue
    html = gzip.decompress(path.read_bytes()).decode("utf-8", errors="replace")
    url = gt.get("url", "")
    state = page_state(html, url)
    availability = gt.get("availability") or []
    availability = availability if isinstance(availability, list) else [availability]
    if availability:
        stock_items.append((ST_KIND, state, ST_TEXT, ST_OPTIONS))
        stock_truth.append("false" if "OutOfStock" in availability else "true")
    prices = [parse_amount(v) for v in (gt.get("price") or [])]
    prices = [v for v in prices if v]
    found = price_candidates(parse_page(html).blocks)[:MAX_PRICE_OPTIONS]   # page order, as a scraper sees them
    gold = [i for i, c in enumerate(found) if c["value"] and any(abs(c["value"] - v) < 0.005 for v in prices)]
    if not prices:
        missing["no_price_label"] += 1
    elif len(found) < 2:
        missing["fewer_than_2_price_candidates"] += 1
    elif not gold:
        missing["true_price_not_in_first_8_candidates"] += 1
    else:
        options = [{"id": f"p{i}", "text": c["text"]} for i, c in enumerate(found)]
        price_items.append(("choice", state, PRICE_QUESTION_TEXT, options))
        price_gold.append(set(gold))

zyte = {"pages": len(truth_by_id), "not_scored": dict(missing)}
st_ids = [o["id"] for o in ST_OPTIONS]
pairs = []
for probs, want in zip(score(stock_items), stock_truth):
    best = max(range(len(probs)), key=probs.__getitem__)
    pairs.append((st_ids[best] == want, probs[best]))
zyte["in_stock"] = summary(pairs)
zyte["in_stock"]["out_of_stock_pages"] = stock_truth.count("false")
zyte["in_stock"]["majority_baseline"] = round(stock_truth.count("true") / len(stock_truth), 4) if stock_truth else None
oos = [ok for (ok, _), want in zip(pairs, stock_truth) if want == "false"]
zyte["in_stock"]["out_of_stock_found"] = f"{sum(oos)}/{len(oos)}"
pairs = []
for probs, gold in zip(score(price_items), price_gold):
    best = max(range(len(probs)), key=probs.__getitem__)
    pairs.append((best in gold, probs[best]))
zyte["price_field"] = summary(pairs)
zyte["price_field"]["random_pick_baseline"] = (
    round(sum(len(g) / len(i[3]) for g, i in zip(price_gold, price_items)) / len(price_items), 4) if price_items else None)
say(json.dumps(zyte, indent=1))

## 3. Review sample and save


In [ ]:
rng = random.Random(SEED)
review = []
for p in rng.sample(pages, min(REVIEW_PAGES, len(pages))):
    review.append({"url": p["url"], "title": p["state"]["title"], "text": p["state"]["text"][:400],
                   "model_page_type": p["page_type"], "confidence": round(p["page_type_conf"], 3),
                   "schema_label": truth(p["rows"]["page_type"]) if "page_type" in p["rows"] else None})
(OUT / "review-sample.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in review),
                                         encoding="utf-8")
result = {"run": RUN_NAME, "random_web": web, "zyte_human_labels": zyte}
(OUT / "report.json").write_text(json.dumps(result, indent=1), encoding="utf-8")
hf.upload_folder(repo_id=HF_REPO, folder_path=str(OUT), path_in_repo=f"{RUN_NAME}/realworld")
say("uploaded to", f"{HF_REPO}/{RUN_NAME}/realworld")